# Udemy Course Level Classification

A classic end-to-end classification notebook converted from a RapidMiner process.

**Objective:** Predict the course `level` using:
- `content_duration`
- `num_lectures`
- `num_subscribers`
- `subject`

**Model:** Decision Tree Classifier  
**Validation:** 10-fold stratified cross-validation + final 20% holdout test


## 1. Import libraries

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 2001


## 2. Load the CSV file

In [2]:
# Works when the notebook is opened from the project folder.
DATA_PATH = Path('data/udemy_courses.csv')

# Fallback for environments that run the notebook from the notebooks folder.
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/udemy_courses.csv')

df_raw = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df_raw.shape}')
df_raw.head()


Dataset shape: (3672, 9)


,course_id,price,num_subscribers,num_reviews,num_lectures,level,content_duration,published_timestamp,subject
0,1070968,200.0,2147.0,23.0,51.0,All Levels,1.5,2017-01-18T20:58:58Z,Business Finance
1,1113822,75.0,2792.0,923.0,274.0,All Levels,39.0,2017-03-09T16:34:20Z,Business Finance
2,1006314,45.0,2174.0,74.0,51.0,Intermediate Level,2.5,2016-12-19T19:26:30Z,Business Finance
3,1210588,95.0,2451.0,11.0,36.0,All Levels,3.0,2017-05-30T20:07:24Z,Business Finance
4,1011058,200.0,1276.0,45.0,26.0,Intermediate Level,2.0,2016-12-13T14:57:18Z,Business Finance


## 3. Initial data inspection

In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3672 entries, 0 to 3671
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   course_id            3672 non-null   int64  
 1   price                3640 non-null   float64
 2   num_subscribers      3640 non-null   float64
 3   num_reviews          3640 non-null   float64
 4   num_lectures         3640 non-null   float64
 5   level                3640 non-null   object 
 6   content_duration     3640 non-null   float64
 7   published_timestamp  3640 non-null   object 
 8   subject              3640 non-null   object 
dtypes: float64(5), int64(1), object(3)
memory usage: 258.3+ KB


In [4]:
df_raw.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
course_id,3672.0,NaN,NaN,NaN,676452.240196,343061.144774,8324.0,408084.5,688558.0,961538.5,1282064.0
price,3640.0,NaN,NaN,NaN,66.105769,61.25603,0.0,20.0,45.0,95.0,200.0
num_subscribers,3640.0,NaN,NaN,NaN,3213.241758,9550.925641,0.0,106.0,900.5,2558.0,268923.0
num_reviews,3640.0,NaN,NaN,NaN,157.573901,940.220286,0.0,4.0,18.0,68.0,27445.0
num_lectures,3640.0,NaN,NaN,NaN,39.987637,50.504061,0.0,15.0,25.0,45.0,779.0
level,3640,4,All Levels,1905,NaN,NaN,NaN,NaN,NaN,NaN,NaN
content_duration,3640.0,NaN,NaN,NaN,4.104771,6.069912,0.0,1.0,2.0,4.5,78.5
published_timestamp,3640,3634,2017-04-23T16:19:01Z,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
subject,3640,4,Business Finance,1192,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
quality_summary = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'missing_values': df_raw.isna().sum(),
    'missing_pct': (df_raw.isna().mean() * 100).round(2),
    'unique_values': df_raw.nunique(dropna=True),
})
quality_summary

,dtype,missing_values,missing_pct,unique_values
course_id,int64,0,0.00,3666
price,float64,32,0.87,38
num_subscribers,float64,32,0.87,2173
num_reviews,float64,32,0.87,511
num_lectures,float64,32,0.87,227
level,object,32,0.87,4
content_duration,float64,32,0.87,105
published_timestamp,object,32,0.87,3634
subject,object,32,0.87,4


## 4. Select the RapidMiner attributes

In [6]:
selected_columns = [
    'content_duration',
    'level',
    'num_lectures',
    'num_subscribers',
    'subject',
]

df = df_raw[selected_columns].copy()
print(f'Selected dataset shape: {df.shape}')
df.head()


Selected dataset shape: (3672, 5)


,content_duration,level,num_lectures,num_subscribers,subject
0,1.5,All Levels,51.0,2147.0,Business Finance
1,39.0,All Levels,274.0,2792.0,Business Finance
2,2.5,Intermediate Level,51.0,2174.0,Business Finance
3,3.0,All Levels,36.0,2451.0,Business Finance
4,2.0,Intermediate Level,26.0,1276.0,Business Finance


## 5. Clean the data

In [7]:
# The source file contains completely empty rows. A row without a target
# cannot be used for supervised classification, so it is removed.
rows_before = len(df)
df = df.dropna(subset=['level']).copy()

# Reproduce the RapidMiner filter: num_subscribers < 250000.
df = df[df['num_subscribers'] < 250_000].copy()

print(f'Rows before cleaning: {rows_before}')
print(f'Rows after cleaning:  {len(df)}')
print(f'Rows removed:         {rows_before - len(df)}')


Rows before cleaning: 3672
Rows after cleaning:  3639
Rows removed:         33


In [8]:
df.isna().sum()

content_duration    0
level               0
num_lectures        0
num_subscribers     0
subject             0
dtype: int64

## 6. Exploratory data analysis

In [9]:
class_counts = df['level'].value_counts()
class_percentages = (df['level'].value_counts(normalize=True) * 100).round(2)

class_distribution = pd.DataFrame({
    'count': class_counts,
    'percentage': class_percentages,
})
class_distribution

,count,percentage
level,,
All Levels,1904,52.32
Beginner Level,1268,34.84
Intermediate Level,409,11.24
Expert Level,58,1.59


In [10]:
ax = class_counts.sort_values().plot(kind='barh', figsize=(8, 4))
ax.set_title('Target Class Distribution')
ax.set_xlabel('Number of Courses')
ax.set_ylabel('Course Level')
plt.tight_layout()
plt.show()


In [11]:
subject_level_table = pd.crosstab(
    df['subject'],
    df['level'],
    normalize='index'
).round(3)
subject_level_table

level,All Levels,Beginner Level,Expert Level,Intermediate Level
subject,,,,
Business Finance,0.581,0.285,0.026,0.107
Graphic Design,0.497,0.405,0.008,0.090
Musical Instruments,0.396,0.449,0.011,0.145
Web Development,0.549,0.328,0.013,0.111


## 7. Define features and target

In [12]:
X = df.drop(columns='level')
y = df['level']

numeric_features = [
    'content_duration',
    'num_lectures',
    'num_subscribers',
]
categorical_features = ['subject']

print('Feature matrix:', X.shape)
print('Target vector:', y.shape)


Feature matrix: (3639, 4)
Target vector: (3639,)


## 8. Train-test split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print('Training set:', X_train.shape)
print('Test set:    ', X_test.shape)
print() 
print('Training class proportions:')
print(y_train.value_counts(normalize=True).round(3))


Training set: (2911, 4)
Test set:     (728, 4)

Training class proportions:
level
All Levels            0.523
Beginner Level        0.348
Intermediate Level    0.112
Expert Level          0.016
Name: proportion, dtype: float64


## 9. Build the preprocessing pipeline

- Numeric missing values: median imputation
- Categorical missing values: most-frequent imputation
- Categorical encoding: one-hot encoding

All transformations are fitted **only on the training data** through the scikit-learn pipeline.


In [14]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical', categorical_transformer, categorical_features),
    ]
)


## 10. Establish a baseline model

In [15]:
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent')),
])

baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
print(f'Baseline holdout accuracy: {baseline_accuracy:.3f}')


Baseline holdout accuracy: 0.523


## 11. Build the Decision Tree model

RapidMiner used gain ratio, pre-pruning and pruning. Scikit-learn does not implement gain ratio or RapidMiner's confidence-based pruning directly. The closest standard implementation uses entropy with the same main depth and sample-size constraints.


In [16]:
decision_tree = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    random_state=RANDOM_STATE,
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', decision_tree),
])
model


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differen

## 12. Evaluate with 10-fold cross-validation

In [17]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'macro_f1': 'f1_macro',
}

cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
)

cv_summary = pd.DataFrame({
    'metric': ['Accuracy', 'Balanced Accuracy', 'Macro F1'],
    'mean': [
        cv_results['test_accuracy'].mean(),
        cv_results['test_balanced_accuracy'].mean(),
        cv_results['test_macro_f1'].mean(),
    ],
    'std': [
        cv_results['test_accuracy'].std(),
        cv_results['test_balanced_accuracy'].std(),
        cv_results['test_macro_f1'].std(),
    ],
}).round(3)

cv_summary

,metric,mean,std
0,Accuracy,0.523,0.028
1,Balanced Accuracy,0.287,0.026
2,Macro F1,0.274,0.033


## 13. Train the final model

In [18]:
model.fit(X_train, y_train)
print('Final model fitted on the complete training set.')

Final model fitted on the complete training set.


## 14. Evaluate on the holdout test set

In [19]:
y_pred = model.predict(X_test)

test_metrics = pd.Series({
    'Accuracy': accuracy_score(y_test, y_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
    'Macro F1': f1_score(y_test, y_pred, average='macro'),
}).round(3)

test_metrics

Accuracy             0.533
Balanced Accuracy    0.298
Macro F1             0.294
dtype: float64

In [20]:
report = classification_report(
    y_test,
    y_pred,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).T.round(3)

,precision,recall,f1-score,support
All Levels,0.567,0.785,0.659,381.000
Beginner Level,0.463,0.323,0.381,254.000
Expert Level,0.000,0.000,0.000,11.000
Intermediate Level,0.333,0.085,0.136,82.000
accuracy,0.533,0.533,0.533,0.533
macro avg,0.341,0.298,0.294,728.000
weighted avg,0.496,0.533,0.493,728.000


## 15. Confusion matrix

In [21]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    xticks_rotation=30,
    cmap=None,
    ax=ax,
)
ax.set_title('Decision Tree Confusion Matrix')
plt.tight_layout()
plt.show()


## 16. Feature importance

In [22]:
fitted_preprocessor = model.named_steps['preprocessor']
fitted_tree = model.named_steps['classifier']

feature_names = fitted_preprocessor.get_feature_names_out()
feature_importance = (
    pd.DataFrame({
        'feature': feature_names,
        'importance': fitted_tree.feature_importances_,
    })
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

feature_importance.head(15)

,feature,importance
0,numeric__num_subscribers,0.402928
1,numeric__num_lectures,0.289540
2,numeric__content_duration,0.230588
3,categorical__subject_Musical Instruments,0.025786
4,categorical__subject_Graphic Design,0.023098
5,categorical__subject_Web Development,0.019602
6,categorical__subject_Business Finance,0.008457


In [23]:
top_features = feature_importance.head(15).sort_values('importance')
ax = top_features.plot(
    x='feature',
    y='importance',
    kind='barh',
    legend=False,
    figsize=(9, 5),
)
ax.set_title('Top 15 Decision Tree Feature Importances')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()


## 17. Visualize the first levels of the tree

In [24]:
fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(
    fitted_tree,
    feature_names=feature_names,
    class_names=fitted_tree.classes_,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=7,
    ax=ax,
)
ax.set_title('Decision Tree Preview — First Four Levels')
plt.tight_layout()
plt.show()


## 18. Inspect predictions

In [25]:
prediction_results = X_test.copy()
prediction_results['actual_level'] = y_test
prediction_results['predicted_level'] = y_pred
prediction_results['correct_prediction'] = (
    prediction_results['actual_level'] == prediction_results['predicted_level']
)

prediction_results.head(15)

,content_duration,num_lectures,num_subscribers,subject,actual_level,predicted_level,correct_prediction
2532,6.5,64.0,1050.0,Web Development,All Levels,All Levels,True
3000,4.0,24.0,4447.0,Web Development,Intermediate Level,All Levels,False
2921,4.0,57.0,716.0,Web Development,Beginner Level,All Levels,False
455,2.0,9.0,0.0,Business Finance,Beginner Level,Intermediate Level,False
444,6.5,31.0,4770.0,Business Finance,Beginner Level,Intermediate Level,False
380,4.0,20.0,2715.0,Business Finance,Expert Level,All Levels,False
2043,1.5,21.0,3312.0,Musical Instruments,Intermediate Level,Beginner Level,False
1915,0.5,12.0,4967.0,Musical Instruments,Beginner Level,All Levels,False
2966,7.0,55.0,2134.0,Web Development,All Levels,All Levels,True
1180,3.0,72.0,612.0,Business Finance,Beginner Level,All Levels,False


## 19. Conclusion

- The dataset is strongly imbalanced toward `All Levels` and `Beginner Level`.
- Accuracy alone can therefore be misleading; balanced accuracy and macro F1 are also reported.
- The holdout test set is evaluated separately after cross-validation.
- The model is useful as a clear classification demonstration, but the available features provide limited separation for the rare `Expert Level` and `Intermediate Level` classes.
